# Spatial Filtering, PSNR Metric & Graded Assignment (TP 2 — Exercise 2)

## Assignment Objectives
1. Load `im1` (original), `im2` (Gaussian noise) and `im3` (impulsive noise).
2. $3\times3$ **mean filter**: OpenCV (`cv2.blur`) vs. the from-scratch `filtreMoyenne`.
3. $3\times3$ **median filter** on `im3` and `im2`: OpenCV vs. `filtreMedian`.
4. Define and validate the **PSNR / MSE** metric (`cv2.PSNR` vs. `calculate_psnr`).
5. Full **filter benchmark** on `im2`/`im3` (mean, Gaussian, median, bilateral) with PSNR and runtimes.
6. **Graded (a):** accelerating a large mean filter — separable 1D passes and the $O(1)$/pixel **integral image**.
7. **Graded (b):** adaptive mean filtering — the course formula, an impulse-robust variant, and the local MMSE (Lee) filter.

## Mathematical Principles & Intuition
### PSNR — Peak Signal-to-Noise Ratio
$$\mathrm{MSE} = \frac{1}{HW}\sum_{y,x}\big(I_{ref}(y,x)-I_{test}(y,x)\big)^2, \qquad
\mathrm{PSNR} = 10\log_{10}\!\left(\frac{255^2}{\mathrm{MSE}}\right)$$
PSNR is logarithmic: +20 dB means the MSE dropped 100x. Identical images give
$\mathrm{MSE}=0$ and, by convention in this repository, $\mathrm{PSNR}=+\infty$.

### Mean vs. median
- Mean filter: $C(y,x) = \frac{1}{k^2}\sum_{(i,j)\in \mathcal{N}_k} I(i,j)$ — a low-pass
  that blurs edges as hard as noise.
- Median filter: the middle order statistic of the window. An isolated 0/255
  impulse can never be the median of a $3\times3$ window — impulses vanish,
  and edges (step discontinuities) survive.
- Bilateral filter: Gaussian weights in **space** *and* **intensity**,
  $w = \exp(-\frac{\|p-q\|^2}{2\sigma_s^2})\,\exp(-\frac{(I(p)-I(q))^2}{2\sigma_r^2})$ — smoothing that respects edges.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Step 1: Deterministic Inputs (`im1`, `im2`, `im3`)
`im2`/`im3` are loaded from the TP2-Exo1 outputs when available and otherwise
regenerated with the same seeds — either way the filtering experiments run on
bit-identical inputs.


In [ ]:
from TAI.utils.metrics import calculate_psnr
from TAI.utils.noise import add_gaussian_noise, sp_noise

im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)

gauss_path = OUT_DIR / "tp2_exo1_cameraman_gauss_var100.bmp"
sp_path = OUT_DIR / "tp2_exo1_cameraman_salt_pepper_p10.bmp"

if gauss_path.exists():
    im2 = cv2.imread(str(gauss_path), cv2.IMREAD_GRAYSCALE)
else:
    im2 = add_gaussian_noise(im1, variance=100.0, seed=42)
    cv2.imwrite(str(gauss_path), im2)

if sp_path.exists():
    im3 = cv2.imread(str(sp_path), cv2.IMREAD_GRAYSCALE)
else:
    im3 = sp_noise(im1, prob=0.10, seed=42)
    cv2.imwrite(str(sp_path), im3)

print("Loaded images:", im1.shape, im2.shape, im3.shape)
print(f"PSNR(im1, im2) = {calculate_psnr(im1, im2):.2f} dB | PSNR(im1, im3) = {calculate_psnr(im1, im3):.2f} dB")


---
## Step 2: Mean / Median / Bilateral — OpenCV vs. From-Scratch
The three from-scratch implementations live in `TAI/utils/filters.py`
(`filtreMoyenne`, `filtreMedian`, `filtrebilateral`) — one canonical,
unit-tested source shared by all notebooks.


In [ ]:
from TAI.utils.filters import filtreMedian, filtreMoyenne, filtrebilateral

# 3x3 mean filter: OpenCV vs. from-scratch
imf2 = cv2.blur(im2, (3, 3))
imfp2 = filtreMoyenne(im2, 3)

# 3x3 median filter on im3 (S&P) and im2 (Gaussian)
imf3 = cv2.medianBlur(im3, 3)
imfm2 = cv2.medianBlur(im2, 3)
imfmp = filtreMedian(im3, 3)

# PSNR validation: cv2.PSNR vs. calculate_psnr
pairs = [
    ("PSNR(im1, im2  Gaussian noise)", im1, im2),
    ("PSNR(im1, imf2 mean 3x3 on im2)", im1, imf2),
    ("PSNR(im1, imfp2 mean custom on im2)", im1, imfp2),
    ("PSNR(im1, im3  S&P noise)", im1, im3),
    ("PSNR(im1, imf3 median 3x3 on im3)", im1, imf3),
    ("PSNR(im1, imfmp median custom on im3)", im1, imfmp),
    ("PSNR(im1, imfm2 median 3x3 on im2)", im1, imfm2),
]
print(f"{'Comparison':38s} | {'cv2.PSNR':>10s} | {'calculate_psnr':>14s}")
print("-" * 68)
for label, ref, test in pairs:
    print(f"{label:38s} | {cv2.PSNR(ref, test):8.3f} dB | {calculate_psnr(ref, test):12.3f} dB")

show_images([
    ("im1 - Original", im1),
    ("im2 - Gaussian noise", im2),
    ("Mean 3x3 on im2 (imf2)", imf2),
    ("im3 - Salt & pepper", im3),
    ("Median 3x3 on im3 (imf3)", imf3),
    ("Median 3x3 on im2 (imfm2)", imfm2),
], cols=3)


In [ ]:
# Interior-pixel agreement between OpenCV and the from-scratch implementations
r = 1  # 3x3 window -> ignore a 1-pixel border (different border conventions)
interior = lambda a, b, r: np.array_equal(a[r:-r, r:-r], b[r:-r, r:-r])
border_max = lambda a, b: int(np.max(np.abs(a.astype(np.int16) - b.astype(np.int16))))

print(f"{'Filter pair':42s} | {'interior ==':>11s} | {'border max diff':>15s}")
print("-" * 74)
print(f"{'cv2.blur(3,3) vs filtreMoyenne(im2, 3)':42s} | {str(interior(imf2, imfp2, r)):>11s} | {border_max(imf2, imfp2):>15d}")
print(f"{'cv2.medianBlur(3) vs filtreMedian(im3,3)':42s} | {str(interior(imf3, imfmp, r)):>11s} | {border_max(imf3, imfmp):>15d}")

bil_cv = cv2.bilateralFilter(im2, 7, 40, 40)
bil_my = filtrebilateral(im2, 7, 2.0, 40.0)
print(f"{'PSNR cv2.bilateral vs filtrebilateral':42s} | {calculate_psnr(bil_cv, bil_my):10.2f} dB (similar behaviour)")


---
## Step 3: Full Filter Benchmark (PSNR + runtime on `im2` and `im3`)


In [ ]:
filters = [
    ("Mean 3x3", lambda img: cv2.blur(img, (3, 3))),
    ("Mean 5x5", lambda img: cv2.blur(img, (5, 5))),
    ("Mean 7x7", lambda img: cv2.blur(img, (7, 7))),
    ("Gaussian 3x3, sig=2.0", lambda img: cv2.GaussianBlur(img, (3, 3), 2.0)),
    ("Gaussian 3x3, sig=1.5", lambda img: cv2.GaussianBlur(img, (3, 3), 1.5)),
    ("Gaussian 3x3, sig=1.0", lambda img: cv2.GaussianBlur(img, (3, 3), 1.0)),
    ("Gaussian 3x3, sig=0.5", lambda img: cv2.GaussianBlur(img, (3, 3), 0.5)),
    ("Gaussian 5x5, sig=2.0", lambda img: cv2.GaussianBlur(img, (5, 5), 2.0)),
    ("Gaussian 5x5, sig=1.5", lambda img: cv2.GaussianBlur(img, (5, 5), 1.5)),
    ("Gaussian 5x5, sig=1.0", lambda img: cv2.GaussianBlur(img, (5, 5), 1.0)),
    ("Gaussian 5x5, sig=0.5", lambda img: cv2.GaussianBlur(img, (5, 5), 0.5)),
    ("Median 3x3", lambda img: cv2.medianBlur(img, 3)),
    ("Median 5x5", lambda img: cv2.medianBlur(img, 5)),
    ("Median 7x7", lambda img: cv2.medianBlur(img, 7)),
    ("Bilateral OpenCV (d=7,40,40)", lambda img: cv2.bilateralFilter(img, 7, 40, 40)),
    ("Bilateral custom (7, 2.0, 40)", lambda img: filtrebilateral(img, 7, 2.0, 40.0)),
]

print(f"Baselines -> im2 (Gaussian): {calculate_psnr(im1, im2):.2f} dB | im3 (S&P): {calculate_psnr(im1, im3):.2f} dB\n")
print(f"{'Filter':30s} | {'PSNR im2 (dB)':>13s} | {'time im2 (ms)':>13s} | {'PSNR im3 (dB)':>13s} | {'time im3 (ms)':>13s}")
print("-" * 92)

best_im2, best_im3 = (-1.0, ""), (-1.0, "")
for name, fn in filters:
    t0 = time.perf_counter()
    out2 = fn(im2)
    dt2 = (time.perf_counter() - t0) * 1000.0
    p2 = calculate_psnr(im1, out2)

    t0 = time.perf_counter()
    out3 = fn(im3)
    dt3 = (time.perf_counter() - t0) * 1000.0
    p3 = calculate_psnr(im1, out3)

    best_im2 = (p2, name) if p2 > best_im2[0] else best_im2
    best_im3 = (p3, name) if p3 > best_im3[0] else best_im3
    print(f"{name:30s} | {p2:13.3f} | {dt2:13.3f} | {p3:13.3f} | {dt3:13.3f}")

print(f"\nBest on im2 (Gaussian)     : {best_im2[1]} ({best_im2[0]:.3f} dB)")
print(f"Best on im3 (Salt & pepper): {best_im3[1]} ({best_im3[0]:.3f} dB)")


---
## Graded Assignment (a): Accelerating a Large Mean Filter

For a $k \times k$ mask on an $H \times W$ image:

| Approach | Complexity | Principle |
|---|---|---|
| Naive nested loops | $O(HWk^2)$ | sum the $k^2$ neighbours per pixel |
| Separable 1D passes | $O(HW \cdot 2k)$ | a box kernel is rank-1: horizontal pass then vertical pass |
| Integral image (SAT) | $O(HW)$ — independent of $k$ | 4 table lookups per pixel: $\mathrm{sum} = SAT(b,r) - SAT(t,r) - SAT(b,l) + SAT(t,l)$ — $(t,l)$ and $(b,r)$ are the window's corners in the exclusive prefix-sum convention |

All three live in `TAI/utils/filters.py` (`filtreMoyenne_naif_boucles`,
`filtreMoyenne_separable`, `filtreMoyenne_integrale`).


In [ ]:
from TAI.utils.filters import (
    filtreMoyenne_integrale,
    filtreMoyenne_naif_boucles,
    filtreMoyenne_separable,
)

print(f"{'Size':>8s} | {'Naive (ms)':>11s} | {'Separable (ms)':>15s} | {'Integral (ms)':>14s} | {'Speed-up':>9s} | {'Max diff':>9s}")
print("-" * 84)
for k in (7, 15, 25, 35):
    t0 = time.perf_counter()
    ref = filtreMoyenne_naif_boucles(im2, k)
    dt_naif = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    res_sep = filtreMoyenne_separable(im2, k)
    dt_sep = (time.perf_counter() - t0) * 1000.0

    t0 = time.perf_counter()
    res_int = filtreMoyenne_integrale(im2, k)
    dt_int = (time.perf_counter() - t0) * 1000.0

    max_diff = int(np.max(np.abs(ref.astype(np.int16) - res_int.astype(np.int16))))
    speedup = dt_naif / max(dt_int, 1e-6)
    print(f"{f'{k}x{k}':>8s} | {dt_naif:11.2f} | {dt_sep:15.3f} | {dt_int:14.3f} | {speedup:8.1f}x | {max_diff:9d}")


---
## Graded Assignment (b): Adaptive Mean Filtering

1. **Course formula (slide 42):**
   $$C[i, j] = \frac{\sum_{k=1}^{L} w(|a_k - A[i,j]|)\, a_k}{\sum_{k=1}^{L} w(|a_k - A[i,j]|)}, \qquad w(x) = \begin{cases} 1 & |x| \le t \\ 0 & \text{otherwise} \end{cases}$$
   Only neighbours within a radiometric distance $t$ of the centre are averaged — edges (large $|a_k - A|$) are excluded, so the filter adapts to local contrast.
2. **Impulse-robust variant:** when the centre is *isolated* (fewer than ~3 similar neighbours — the typical S&P signature), substitute the local **median**.
3. **Local MMSE (Lee) filter:** the smoothing gain adapts to the ratio between noise variance $\sigma_n^2$ and local variance $\sigma_L^2$ — flat areas are strongly smoothed, textured areas are preserved.


In [ ]:
from TAI.utils.filters import filtreMoyenneAdaptative, filtreMoyenneAdaptative_MMSE

comp_rows = [
    ("Noisy (reference)", im2, im3),
    ("Mean 3x3", cv2.blur(im2, (3, 3)), cv2.blur(im3, (3, 3))),
    ("Gaussian 3x3 sig=1.0", cv2.GaussianBlur(im2, (3, 3), 1.0), cv2.GaussianBlur(im3, (3, 3), 1.0)),
    ("Median 3x3", cv2.medianBlur(im2, 3), cv2.medianBlur(im3, 3)),
    ("Bilateral (d=7,40,40)", cv2.bilateralFilter(im2, 7, 40, 40), cv2.bilateralFilter(im3, 7, 40, 40)),
    ("Adaptive mean course (5x5, t=30)", filtreMoyenneAdaptative(im2, 5, 30, False),
     filtreMoyenneAdaptative(im3, 5, 30, False)),
    ("Adaptive mean robust (5x5, t=30)", filtreMoyenneAdaptative(im2, 5, 30, True),
     filtreMoyenneAdaptative(im3, 5, 30, True)),
    ("Adaptive MMSE Lee (5x5)", filtreMoyenneAdaptative_MMSE(im2, 5, 100.0),
     filtreMoyenneAdaptative_MMSE(im3, 5, 100.0)),
]

print(f"{'Method':40s} | {'PSNR im2 (Gauss)':>16s} | {'PSNR im3 (S&P)':>14s}")
print("-" * 76)
for label, r2, r3 in comp_rows:
    print(f"{label:40s} | {calculate_psnr(im1, r2):14.3f} dB | {calculate_psnr(im1, r3):12.3f} dB")

show_images([
    ("Adaptive course on im2", comp_rows[5][1]),
    ("Adaptive MMSE on im2", comp_rows[7][1]),
    ("Adaptive course on im3", comp_rows[5][2]),
    ("Adaptive robust on im3", comp_rows[6][2]),
], cols=4)


## Analysis & Key Takeaways
- **PSNR validation:** `calculate_psnr` matches `cv2.PSNR` to the third decimal on every pair; identical images return $+\infty$ by the repository-wide convention documented in `TAI/utils/metrics.py`.
- **Gaussian noise (`im2`):** linear low-pass filters (mean/Gaussian) trade detail for noise reduction; the bilateral filter and the adaptive mean keep edges far better at similar PSNR. The MMSE/Lee filter adapts smoothing to the local variance and is the best linear-MMSE compromise.
- **Impulsive noise (`im3`):** the median filter is untouchable — an impulse can never be the median of a $3\times3$ window, so PSNR jumps ~4 dB while mean filters *spread* the impulses into grey smudges.
- **Graded (a):** the separable pass divides the per-pixel cost by $k/2$; the integral image makes it *constant* — a 35x35 filter runs as fast as a 3x3 one (only 4 lookups per pixel), with identical outputs up to rounding.
- **Graded (b):** the adaptive course formula preserves edges by construction; adding the median fallback makes it robust to impulses; the Lee/MMSE filter is optimal for additive Gaussian noise in the MMSE sense.
